In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go 
import os
import pyodbc

In [5]:
# import FinanceLib as fl
# style.use('ggplot')

%matplotlib inline
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
# fname  = 'C:/Users/40KravchukPV.REGION/Documents/Courses/FinanceProject/config.txt'
fname  = 'C:/Dev/Python/FinanceProject/config.txt'
config_dict = fl.ReadConnConfig(fname)
con = pyodbc.connect(driver = config_dict['DRIVER'],server = config_dict['SERVER'], port = config_dict['PORT'], database = config_dict['DATABASE'], UID = config_dict['UID'], PWD = config_dict['PWD'], autocommit=True)

In [ ]:
os.environ['HTTP_PROXY']="10.0.63.134:3128"

In [6]:
# confirmed_global
confirmed_global='https://raw.githubusercontent.com/CSSEGISandData/COVID-19/master/csse_covid_19_data/csse_covid_19_time_series/time_series_covid19_confirmed_global.csv'
# confirmed_global='C:/Users/40KravchukPV.REGION/Documents/Courses/FinanceProject/time_series_covid19_confirmed_global.csv'

covid19_confirmed=pd.read_csv(confirmed_global,index_col='Country/Region')

confirmed_latest = covid19_confirmed.T.index.values[-1]

df_grouped_conf=covid19_confirmed.groupby('Country/Region').sum()

df_confirmed=df_grouped_conf.sort_values(by=confirmed_latest,ascending=False).head(10).drop(['Lat', 'Long'],axis=1).T

In [8]:
def multi_plot(df, title, addAll = True):
    fig = go.Figure()

    for column in df.columns.to_list():
        fig.add_trace(
            go.Scatter(
                x = df.index,
                y = df[column],
                name = column
            )
        )

    button_all = dict(label = 'All',
                      method = 'update',
                      args = [{'visible': df.columns.isin(df.columns),
                               'title': 'All',
                               'showlegend':True}])

    def create_layout_button(column):
        return dict(label = column,
                    method = 'update',
                    args = [{'visible': df.columns.isin([column]),
                             'title': column,
                             'showlegend': True}])

    fig.update_layout(
        updatemenus=[go.layout.Updatemenu(
            active = 0,
            buttons = ([button_all] * addAll) + list(df.columns.map(lambda column: create_layout_button(column)))
            )
        ],
         yaxis_type="log"       
    )
    # Update remaining layout properties
    fig.update_layout(
        title_text=title,
        height=800
        
    )
   
    fig.show()

In [9]:
multi_plot(df_confirmed, title="Logarithmic COVID-19 time series total confirmed by country")       

In [11]:
tickers_list = ['ES', 'AAPL']
df_input = fl.GetStockQuoteFromDB(con, tickers_list, IsDtIndex = 1, IsStockIndex = 0, DateFrom = 'NULL', DateTo = 'NULL')

In [12]:
df_input.head()

,Stock,OpenValue,HighValue,LowValue,CloseValue,AdjClose,Volume,LoadDt
Dt,,,,,,,,
2015-12-31,AAPL,26.752501,26.757500,26.205000,26.315001,24.380093,163649200.0,2021-01-28 19:49:35.147
2015-12-31,ES,51.889999,51.900002,50.549999,51.070000,43.896469,2081200.0,2021-01-28 20:53:03.533
2016-01-04,AAPL,25.652500,26.342501,25.500000,26.337500,24.400942,270597600.0,2021-01-28 19:49:35.160
2016-01-04,ES,50.650002,50.889999,50.230000,50.880001,43.733162,1590300.0,2021-01-28 20:53:03.550
2016-01-05,AAPL,26.437500,26.462500,25.602501,25.677500,23.789471,223164000.0,2021-01-28 19:49:35.177


In [18]:
df_input['Stock'].unique()

array(['AAPL', 'ES'], dtype=object)

In [29]:
pd.Series(df_input['Stock'].unique())

0    AAPL
1      ES
dtype: object

In [19]:
for column in df_input['Stock'].unique():
    print(column)

AAPL
ES


In [39]:
df_input[df_input['Stock'] == 'AAPL']['AdjClose']

Dt
2015-12-31     24.380093
2016-01-04     24.400942
2016-01-05     23.789471
2016-01-06     23.323915
2016-01-07     22.339539
                 ...    
2021-02-01    134.139999
2021-02-02    134.990005
2021-02-03    133.940002
2021-02-04    137.389999
2021-02-05    136.759995
Name: AdjClose, Length: 1284, dtype: float64

In [20]:
df = df_confirmed

In [28]:
type(df.columns)

pandas.core.indexes.base.Index

In [21]:
df.columns.isin(df.columns)

array([ True,  True,  True,  True,  True,  True,  True,  True,  True,
        True])

In [40]:
def multi_plot_stock(df, ColumnTarget, title, addAll = True):
    fig = go.Figure()

    for column in df['Stock'].unique():
        fig.add_trace(
            go.Scatter(
                x = df.index,
                y = df_input[df_input['Stock'] == column][ColumnTarget],
                name = column
            )
        )

    button_all = dict(label = 'All',
                      method = 'update',
                      args = [{'visible': pd.Series(df['Stock'].unique()).isin(df['Stock'].unique()),
                               'title': 'All',
                               'showlegend':True}])

    def create_layout_button(column):
        return dict(label = column,
                    method = 'update',
                    args = [{'visible': pd.Series(df['Stock'].unique()).isin([column]),
                             'title': column,
                             'showlegend': True}])

    fig.update_layout(
        updatemenus=[go.layout.Updatemenu(
            active = 0,
            buttons = ([button_all] * addAll) + list(pd.Series(df['Stock'].unique()).map(lambda column: create_layout_button(column)))
            )
        ],
         yaxis_type="log"       
    )
    # Update remaining layout properties
    fig.update_layout(
        title_text=title,
        height=800
        
    )
   
    fig.show()

In [41]:
multi_plot_stock(df_input, 'AdjClose', title="Stock")   

In [15]:
import plotly.graph_objects as go

import pandas as pd
from datetime import datetime

df = pd.read_csv('https://raw.githubusercontent.com/plotly/datasets/master/finance-charts-apple.csv')

fig = go.Figure(data=[go.Candlestick(x=df['Date'],
                open=df['AAPL.Open'],
                high=df['AAPL.High'],
                low=df['AAPL.Low'],
                close=df['AAPL.Close'])])

fig.show()

In [11]:
import plotly.graph_objects as go

fig = go.Figure(data=go.Bar(y=[2, 3, 1]))
fig.show()


In [13]:
%matplotlib inline

In [14]:
import plotly.io as pio
pio.renderers.default = "notebook_connected"
fig.show()